In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Task 1:
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
df.describe()   # Task 4: Write your code here:

In [ ]:
# Task 1: Write your code here:
# Task 2:
# cheaking for missing value
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
df = df.fillna(df.mean())

In [ ]:
df

In [ ]:
# Task 2: Write your code here:
# 4. Check for duplicates
def check_duplicates(df):

  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
# 3. Do we have categorical columns?
def encode_categorical_columns(df):
    categorical_cols = df.select_dtypes(include=["object"]).columns
    print("Categorical Columns:", list(categorical_cols))

label_encoders = encode_categorical_columns(df)

In [ ]:
df.info()

In [ ]:
from sklearn.preprocessing import LabelEncoder

for col in df.select_dtypes(include=["number"]).columns:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df

In [ ]:
# Task 4: Write your code here:
# Task : feature scaling
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=['number']).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 5: Write your code here:
# Task 6: check for imbalance
print(df['Target'].value_counts())
print(df['Target'].value_counts(normalize=True))    # it is imbalance
df['Target'].hist()

In [ ]:
# Task 1: Write your code here:
# Task 1:
X = df.drop('Target',axis=1)
y = df['Target']


In [ ]:
%pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        loss_function='MultiClass',
        verbose=0,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    f1 = f1_score(y_val, y_pred, average='macro')
    f1_scores.append(f1)



In [ ]:
print(f"Average F1-macro score across folds: {np.mean(f1_scores):.4f}")


In [ ]:
# Task 1: Write your code here:
import pandas as pd

feature_importance = model.get_feature_importance()
feature_names = X.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(
    importance_df['Feature'],
    importance_df['Importance']
)
plt.gca().invert_yaxis()
plt.xlabel('Importance Score')
plt.title('CatBoost Feature Importance')
plt.show()


In [ ]:
# Task 2: Write your code here:
golden_feature = importance_df.iloc[0]

print("Golden Feature:")
print(f"Feature Name: {golden_feature['Feature']}")
print(f"Importance Score: {golden_feature['Importance']:.4f}")


In [ ]:
# Task Bonus: Write your code here: